In [1]:
from pymongo import MongoClient

client = MongoClient("127.0.0.1", 27017)
db = client["api_update"]
col = db["py_candidate_api_update_instances"]
col.estimated_document_count()

76761

In [2]:
from tqdm import tqdm
import pandas as pd
from packaging.version import Version

tqdm.pandas()
commit_pairs_orig = []

for doc in tqdm(col.find({}), total=col.estimated_document_count()):
    commit = doc["commit"]
    version_before = doc["version_before"]
    version_after = doc["version_after"]
    for pair in doc["api_update_pairs"]:
        old_callee = pair["old_callee"]
        old_api = old_callee["full_name"]
        if any(n.startswith("_") for n in old_api.split(".")):
            continue
        new_callee = pair["new_callee"]
        new_api = new_callee["full_name"]
        if any(n.startswith("_") for n in new_api.split(".")):
            continue

        record = [
            doc["package"],
            version_before,
            version_after,
            old_api,
            new_api,
            commit,
        ]
        old_v = Version(version_before)
        new_v = Version(version_after)

        if old_v == new_v:
            continue

        # old_api, -> new_api, up
        # new_api -> old_api, down
        elif old_v < new_v:
            if old_api < new_api:
                rule = [(old_api, new_api), "Up"]
            else:
                rule = [(new_api, old_api), "Down"]

        # old_api -> new_api, down
        # new_api -> old_api, up
        else:
            if old_api < new_api:
                rule = [(old_api, new_api), "Down"]
            else:
                rule = [(new_api, old_api), "Up"]
        commit_pairs_orig.append(record + rule)


commit_pairs_orig = (
    pd.DataFrame(
        commit_pairs_orig,
        columns=[
            "package",
            "version_before",
            "version_after",
            "old_api",
            "new_api",
            "commit",
            "rule",
            "direction",
        ],
    )
    .drop_duplicates()
    .dropna()
)

print(len(commit_pairs_orig), "commit pairs before filtering")
print(
    len(commit_pairs_orig[["package", "rule", "direction"]].drop_duplicates()),
    "rules before filtering",
)
print(
    len(
        commit_pairs_orig[
            ["package", "version_before", "version_after", "old_api", "new_api"]
        ].drop_duplicates()
    ),
    "pairs before filtering",
)

100%|██████████| 76761/76761 [00:17<00:00, 4268.09it/s]


48755 commit pairs before filtering
12301 rules before filtering
40537 pairs before filtering


In [3]:
def ratio_fun(row):
    if row["Down"] > row["Up"]:
        row["ratio"] = row["Down"] / row["Up"]
    else:
        row["ratio"] = row["Up"] / row["Down"]

    return row


def find_naive_error_rules(threshold: int):
    rule_df = (
        commit_pairs_orig.groupby(["package", "rule", "direction"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
        .rename_axis(None, axis=1)
    )
    candidate_error_rules = rule_df[(rule_df["Down"] > 0) & (rule_df["Up"] > 0)]
    print(f"{len(candidate_error_rules)} rules have both Up and Down direction")
    candidate_error_rules = candidate_error_rules.apply(ratio_fun, axis=1).sort_values(
        "ratio", ascending=False
    )
    data = []
    for row in candidate_error_rules.itertuples(index=False):
        if row.ratio < threshold:
            data.append([row.package, row.rule, "Up"])
            data.append([row.package, row.rule, "Down"])
        else:
            if row.Up > row.Down:
                data.append([row.package, row.rule, "Down"])
            else:
                data.append([row.package, row.rule, "Up"])
    print(len(data), "error rules")
    error_rules = pd.DataFrame(data, columns=["package", "rule", "direction"])
    return error_rules


error_rules = find_naive_error_rules(7)

1399 rules have both Up and Down direction
2560 error rules


In [4]:
commit_pairs_filtered = (
    pd.merge(commit_pairs_orig, error_rules, indicator=True, how="left")
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
print(len(commit_pairs_filtered), "commit pairs after filtering")
print(
    len(commit_pairs_filtered[["package", "rule"]].drop_duplicates()),
    "rules after filtering",
)

37999 commit pairs after filtering
9741 rules after filtering


In [5]:
def canonical_pairs(row):
    version_before = row["version_before"]
    version_after = row["version_after"]
    package = row["package"]
    old_api = row["old_api"]
    new_api = row["new_api"]
    commit = row["commit"]
    if Version(version_before) > Version(version_after):
        return pd.Series(
            [package, version_after, version_before, new_api, old_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )
    else:
        return pd.Series(
            [package, version_before, version_after, old_api, new_api, commit],
            index=[
                "package",
                "old_version",
                "new_version",
                "old_api",
                "new_api",
                "commit",
            ],
        )


commit_pairs_full = commit_pairs_filtered.apply(canonical_pairs, axis=1)

In [6]:
rule_freq = (
    commit_pairs_full.groupby(["package", "old_api", "new_api"])["commit"]
    .nunique()
    .reset_index()
    .sort_values("commit", ascending=False, ignore_index=True)
)
num_packages = rule_freq["package"].nunique()
num_releases = len(
    pd.concat(
        [
            commit_pairs_full[["package", "old_version"]].rename(
                columns={"old_version": "version"}
            ),
            commit_pairs_full[["package", "new_version"]].rename(
                columns={"new_version": "version"}
            ),
        ]
    ).drop_duplicates()
)
num_rules = len(rule_freq)
num_commits = commit_pairs_full["commit"].nunique()
print(f"# Packages: {num_packages}")
print(f"# Releases: {num_releases}")
print(f"# Rules: {num_rules}")
print(f"# Commits: {num_commits}")

# Packages: 1366
# Releases: 7690
# Rules: 9741
# Commits: 11814


In [7]:
rules_gte10 = rule_freq[rule_freq["commit"] >= 10]
print(
    f"{len(rules_gte10)} rules in {rules_gte10['package'].nunique()} Python packages with freq >= 10"
)
rules_1to10 = rule_freq[(rule_freq["commit"] > 1) & (rule_freq["commit"] < 10)]
print(
    f"{len(rules_1to10)} rules in {rules_1to10['package'].nunique()} Java packages with freq > 1 and < 10"
)
rules_eq1 = rule_freq[rule_freq["commit"] == 1]
print(
    f"{len(rules_eq1)} rules in {rules_eq1['package'].nunique()} Java packages with freq = 1"
)

343 rules in 73 Python packages with freq >= 10
3679 rules in 578 Java packages with freq > 1 and < 10
5719 rules in 1113 Java packages with freq = 1


In [8]:
from utils import cal_sample_size

population_size = len(rules_1to10) + len(rules_eq1)
sample_size = cal_sample_size(population_size)
sample_size_1to10 = round(sample_size * len(rules_1to10) / population_size)
sample_size_eq1 = round(sample_size * len(rules_eq1) / population_size)
print(f"Sample size for all rules with freq < 10: {sample_size}")
print(f"Sample size for rules with freq > 1 and < 10: {sample_size_1to10}")
print(f"Sample size for rules with freq = 1: {sample_size_eq1}")
rules_gte10.to_excel(
    "../benchmark/final/python_api_update_rules_gte10.xlsx", index=False
)
rules_1to10.sample(sample_size_1to10).to_excel(
    "../benchmark/final/python_api_update_rules_1to10.xlsx", index=False
)
rules_eq1.sample(sample_size_eq1).to_excel(
    "../benchmark/final/python_api_update_rules_eq1.xlsx", index=False
)

Sample size for all rules with freq < 10: 369
Sample size for rules with freq > 1 and < 10: 144
Sample size for rules with freq = 1: 225


In [9]:
rules_gte10_labelled = pd.read_excel(
    "../benchmark/final/python_api_update_rules_gte10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_gte10 = rules_gte10_labelled[rules_gte10_labelled["correct"] == 1]
print(
    f"{len(rules_gte10_labelled)} rules with freq >= 10, {len(correct_rules_gte10)} are correct"
)
print(f"Accuracy: {len(correct_rules_gte10) / len(rules_gte10_labelled):.3f}")

rules_1to10_labelled = pd.read_excel(
    "../benchmark/final/python_api_update_rules_1to10-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_1to10 = rules_1to10_labelled[rules_1to10_labelled["correct"] == 1]
print(
    f"{len(rules_1to10_labelled)} rules with freq (1, 10), {len(correct_rules_1to10)} are correct"
)
print(f"Accuracy: {len(correct_rules_1to10) / len(rules_1to10_labelled):.3f}")

rules_eq1_labelled = pd.read_excel(
    "../benchmark/final/python_api_update_rules_eq1-labelled.xlsx",
    keep_default_na=False,
)
correct_rules_eq1 = rules_eq1_labelled[rules_eq1_labelled["correct"] == 1]
print(
    f"{len(rules_eq1_labelled)} rules with freq = 1, {len(correct_rules_eq1)} are correct"
)
print(f"Accuracy: {len(correct_rules_eq1) / len(rules_eq1_labelled):.3f}")

rules_exact = pd.concat(
    [correct_rules_gte10, correct_rules_1to10, correct_rules_eq1]
).sort_values("commit", ascending=False, ignore_index=True)
print(
    f"{len(rules_exact)} verified rule in total, {rules_exact['package'].nunique()} packages"
)
total_sampled_rules = pd.concat(
    [rules_gte10_labelled, rules_1to10_labelled, rules_eq1_labelled]
).sort_values("commit", ascending=False, ignore_index=True)
total_sampled_rules.to_csv("../benchmark/final/python_labelled_rules.csv", index=False)

343 rules with freq >= 10, 329 are correct
Accuracy: 0.959
144 rules with freq (1, 10), 131 are correct
Accuracy: 0.910
225 rules with freq = 1, 205 are correct
Accuracy: 0.911
665 verified rule in total, 213 packages


In [10]:
estimated_accuracy = (
    len(correct_rules_gte10)
    + len(correct_rules_1to10) / len(rules_1to10_labelled) * len(rules_1to10)
    + len(correct_rules_eq1) / len(rules_eq1_labelled) * len(rules_eq1)
) / len(rule_freq)
print(f"Estimated overall accuracy:", estimated_accuracy)

Estimated overall accuracy: 0.9122792834411252


In [11]:
def get_update_type(row):
    result = [3, 2, 1]
    old_version = row["old_version"].split(".")
    new_version = row["new_version"].split(".")
    min_len = min(len(old_version), len(new_version))
    for i in range(min_len):
        if old_version[i] != new_version[i]:
            row["change"] = (int(old_version[i]), int(new_version[i]))
            row["update_type"] = result[i]
            return row
    row["change"] = (0, int(new_version[i]))
    row["update_type"] = result[min_len]
    return row

In [12]:
commit_pairs_full = commit_pairs_full.apply(get_update_type, axis=1)
commit_pairs_full = commit_pairs_full[
    commit_pairs_full["update_type"]
    == commit_pairs_full.groupby(["package", "old_api", "new_api"])[
        "update_type"
    ].transform("max")
]

In [14]:
commit_pairs_exact = (
    rules_exact[["package", "old_api", "new_api"]]
    .merge(commit_pairs_full)
    .drop_duplicates()
)
pairs_exact = commit_pairs_exact[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
print(
    f"{len(commit_pairs_exact)} commit pairs, {len(pairs_exact)} pairs, {len(rules_exact)} correct verified rules, {rules_exact['package'].nunique()} packages"
)
commit_pairs_exact[
    ["package", "old_version", "old_api", "new_version", "new_api", "commit"]
].to_json(
    "../benchmark/final/python_commit_pairs_exact.json", indent=2, orient="records"
)
pairs_exact.to_json(
    "../benchmark/final/python_api_pairs_exact.json", indent=2, orient="records"
)

11028 commit pairs, 7501 pairs, 665 correct verified rules, 213 packages


In [15]:
new_commit_pairs_full = (
    commit_pairs_full.merge(
        total_sampled_rules[total_sampled_rules["correct"] == 0][
            ["package", "old_api", "new_api"]
        ],
        indicator=True,
        how="left",
    )
    .query('_merge=="left_only"')
    .drop("_merge", axis=1)
)
pairs_full = new_commit_pairs_full[
    ["package", "old_version", "old_api", "new_version", "new_api"]
].drop_duplicates()
rules_all = new_commit_pairs_full[["package", "old_api", "new_api"]].drop_duplicates()
print(
    f"{len(new_commit_pairs_full)} commit pairs, {len(pairs_full)} pairs, {len(rules_all)} rules, {rules_all['package'].nunique()} Python packages after removing incorrect verified rules"
)

new_commit_pairs_full[
    ["package", "old_version", "old_api", "new_version", "new_api", "commit"]
].to_json(
    "../benchmark/final/python_commit_pairs_full.json", indent=2, orient="records"
)
pairs_full.to_json(
    "../benchmark/final/python_api_pairs_full.json", indent=2, orient="records"
)

27983 commit pairs, 22070 pairs, 9694 rules, 1363 Python packages after removing incorrect verified rules


In [16]:
rules_exact.groupby(["package", "old_api"])["new_api"].apply(set).reset_index().to_json(
    "../benchmark/final/python_api_update_rules_exact.json", indent=2, orient="records"
)

In [17]:
pairs_full[["package", "old_api", "new_api"]].drop_duplicates().groupby(
    ["package", "old_api"]
)["new_api"].apply(set).reset_index().to_json(
    "../benchmark/final/python_api_update_rules_full.json", indent=2, orient="records"
)

In [18]:
pairs_exact_sample = (
    pairs_exact.apply(get_update_type, axis=1)
    .groupby(["package", "old_api", "new_api", "change"])
    .sample(1)
)[["package", "old_version", "old_api", "new_version", "new_api"]]
print(f"{len(pairs_exact_sample)} sampled pairs in the exact group")
pairs_exact_sample.to_json(
    "../benchmark/final/python_sampled_api_pairs_exact.json", indent=2, orient="records"
)

pairs_full_sample = (
    pairs_full.apply(get_update_type, axis=1)
    .groupby(["package", "old_api", "new_api", "change"])
    .sample(1)
)[["package", "old_version", "old_api", "new_version", "new_api"]]
print(f"{len(pairs_full_sample)} sampled pairs in the full group")
pairs_full_sample.to_json(
    "../benchmark/final/python_sampled_api_pairs_full.json", indent=2, orient="records"
)

1603 sampled pairs in the exact group
14068 sampled pairs in the full group


In [20]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()


def sample_instances(group: str):
    instances = pd.read_json(f"../benchmark/final/python_update_instances_{group}.json")
    num_pairs = len(
        instances[
            ["package", "old_api", "new_api", "old_version", "new_version"]
        ].drop_duplicates()
    )
    num_rules = len(instances[["package", "old_api", "new_api"]].drop_duplicates())
    print(f"{group} group:")
    print(f"  # Instances: {len(instances)}")
    print(f"  # Pairs: {num_pairs}")
    print(f"  # Rules: {num_rules}")
    sample = (
        instances.progress_apply(get_update_type, axis=1)
        .groupby(["package", "old_api", "new_api", "change"])
        .sample(1, random_state=42)
    )
    print(f"  # Sampled Instances: {len(sample)}")
    sample.to_json(
        f"../benchmark/final/python_sampled_update_instances_{group}.json",
        index=False,
        orient="records",
    )


sample_instances("exact")
sample_instances("full")

exact group:
  # Instances: 52848
  # Pairs: 3602
  # Rules: 464


100%|██████████| 52848/52848 [00:31<00:00, 1662.55it/s]


  # Sampled Instances: 1017
full group:
  # Instances: 84698
  # Pairs: 12698
  # Rules: 6180


100%|██████████| 84698/84698 [00:49<00:00, 1703.08it/s]


  # Sampled Instances: 8980
